In [ ]:
"""Simple CSV interpolator for Sionna RT."""
import pandas as pd
from scipy.interpolate import interp1d
import mitsuba as mi
from typing import Tuple, Callable
import sionna.rt
from sionna.rt import load_scene
from sionna.rt import RadioMaterial

# Functions to extract the data you need for the material
def load_csv_interpolators(permittivity_csv: str, conductivity_csv: str):
    """
    Load CSV data and return a frequency callback function.

    Args:
        permittivity_csv: Path to permittivity CSV (freq_MHz, value)
        conductivity_csv: Path to conductivity CSV (freq_MHz, value)

    Returns:
        Callable that takes frequency [Hz] and returns (permittivity, conductivity)
    """
    # Load CSV data once
    df_per = pd.read_csv(permittivity_csv, header=None, names=["freq_mhz", "value"])
    df_con = pd.read_csv(conductivity_csv, header=None, names=["freq_mhz", "value"])

    # Create interpolators
    per_interp = interp1d(df_per["freq_mhz"], df_per["value"], kind='linear')
    con_interp = interp1d(df_con["freq_mhz"], df_con["value"], kind='linear')

    def callback(frequency_hz: mi.Float) -> Tuple[mi.Float, mi.Float]:
        """Frequency update callback for Sionna RadioMaterial."""
        freq_mhz = frequency_hz / 1e6
        permittivity = mi.Float(float(per_interp(freq_mhz)))
        conductivity = mi.Float(float(con_interp(freq_mhz)))
        return permittivity, conductivity

    return callback

# Wrapper for clarity (refer to sionna for a granular explanation)
def create_custom_material(material_name, per_csv, con_csv):
    print("Creating a custom material")
    #Wraps the RadioMaterial constructor
    new_itu_mat = sionna.rt.RadioMaterial(name=material_name, frequency_update_callback=load_csv_interpolators(per_csv, con_csv))

    return new_itu_mat

In [ ]:
# Update the material definitions to use the updated ground materials
import sys
import sionna.rt

# I ran this in an external project
g2sm = "/home/tingjunlab/Development/geo2sigmap"
sys.path.append(g2sm)
from geo2sigmap.materials.csv_interpolator import create_custom_material

# Create your material definitions
new_wet_ground = create_custom_material(
    "new_wet_ground",
    "/home/tingjunlab/Development/geo2sigmap/research/data/ITU-R_P.527-3_extracted_data/B_wet_ground_per.csv",
    "/home/tingjunlab/Development/geo2sigmap/research/data/ITU-R_P.527-3_extracted_data/B_wet_ground_con.csv",
)
new_medium_dry_ground = create_custom_material(
    "new_medium_dry_ground",
    "/home/tingjunlab/Development/geo2sigmap/research/data/ITU-R_P.527-3_extracted_data/D_medium_dry_ground_per.csv",
    "/home/tingjunlab/Development/geo2sigmap/research/data/ITU-R_P.527-3_extracted_data/D_medium_dry_ground_con.csv",
)
new_very_dry_ground = create_custom_material(
    "new_very_dry_ground",
    "/home/tingjunlab/Development/geo2sigmap/research/data/ITU-R_P.527-3_extracted_data/E_very_dry_ground_per.csv",
    "/home/tingjunlab/Development/geo2sigmap/research/data/ITU-R_P.527-3_extracted_data/E_very_dry_ground_con.csv",
)

In [ ]:
scene_xml_path = "fill\in"

scene = load_scene(scene_xml_path)

# Replace ground material with the custom material you want
# I chose wet ground
ground = scene.get("ground")
ground.radio_material = new_wet_ground

# Replace ALL ITU materials in the scene that might not support 12 GHz
from sionna.rt.radio_materials.itu_material import ITURadioMaterial

for mat_name, mat_obj in list(scene._radio_materials.items()):
    if isinstance(mat_obj, ITURadioMaterial):
        # Replace with appropriate custom material based on name
        if "wet_ground" in mat_name.lower() or "ground" in mat_name.lower():
            print(f"  → Replacing '{mat_name}' with custom wet_ground material")
            scene._radio_materials[mat_name] = new_wet_ground
        elif "dry" in mat_name.lower():
            print(f"  → Replacing '{mat_name}' with custom very_dry_ground material")
            scene._radio_materials[mat_name] = new_very_dry_ground
        else:
            # Default replacement for other ITU materials
            print(f"  → Replacing '{mat_name}' with custom wet_ground material (default)")
            scene._radio_materials[mat_name] = new_wet_ground

# Now it's safe to set the frequency: try it out
scene.frequency = 12e9  # FR3 frequency
print(f"✓ Frequency set to {scene.frequency/1e9} GHz")